**Inventory**

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from pathlib import Path

In [26]:
file_path = Path.cwd().parent.parent/"Data"/"Cleaned"/"Inventory.csv"
df = pd.read_csv(file_path)
df

,Inventory_ID,Product_ID,Warehouse_ID,Date,Opening_Stock,Received_Qty,Sold_Qty,Damaged_Qty,Closing_Stock
0,INV100000,P1051,W109,2025-07-01,2185.0,325.0,672.0,3.0,1835.0
1,INV100001,P1976,W102,2025-08-12,1544.0,593.0,281.0,0.0,1856.0
2,INV100002,P1328,W107,2025-06-17,613.0,542.0,248.0,1.0,906.0
3,INV100003,P1097,W105,2025-01-04,521.0,522.0,372.0,2.0,669.0
4,INV100004,P1160,W110,2025-11-06,1146.0,256.0,340.0,0.0,1062.0
...,...,...,...,...,...,...,...,...,...
365005,INV465005,P1831,W116,2025-11-15,937.0,73.0,213.0,0.0,797.0
365006,INV465006,P1023,W116,2025-05-30,292.0,342.0,469.0,1.0,164.0
365007,INV465007,P1367,W117,2025-05-08,2278.0,262.0,448.0,0.0,2092.0
365008,INV465008,P1995,W104,2025-01-12,1294.0,418.0,37.0,1.0,1674.0


In [27]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365010 entries, 0 to 365009
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Inventory_ID   365010 non-null  object 
 1   Product_ID     365010 non-null  object 
 2   Warehouse_ID   365003 non-null  object 
 3   Date           363184 non-null  object 
 4   Opening_Stock  365010 non-null  float64
 5   Received_Qty   365010 non-null  float64
 6   Sold_Qty       365010 non-null  float64
 7   Damaged_Qty    365010 non-null  float64
 8   Closing_Stock  365010 non-null  float64
dtypes: float64(5), object(4)
memory usage: 25.1+ MB


In [28]:
# Converting to date column
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.month

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365010 entries, 0 to 365009
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Inventory_ID   365010 non-null  object        
 1   Product_ID     365010 non-null  object        
 2   Warehouse_ID   365003 non-null  object        
 3   Date           363184 non-null  datetime64[ns]
 4   Opening_Stock  365010 non-null  float64       
 5   Received_Qty   365010 non-null  float64       
 6   Sold_Qty       365010 non-null  float64       
 7   Damaged_Qty    365010 non-null  float64       
 8   Closing_Stock  365010 non-null  float64       
 9   Month          363184 non-null  float64       
dtypes: datetime64[ns](1), float64(6), object(3)
memory usage: 27.8+ MB


In [30]:
res = df.groupby("Warehouse_ID").agg(
    Quantity_Sold = ("Sold_Qty", "sum"),
    Stock_Left = ("Closing_Stock", "sum"),
).sort_values(by="Stock_Left", ascending=False)
res

,Quantity_Sold,Stock_Left
Warehouse_ID,,
W118,7410881.0,22466848.0
W114,7291647.0,22382837.0
W103,7356631.0,22279452.0
W109,7285869.0,22212591.0
W116,7350104.0,22170138.0
W112,7366934.0,22132392.0
W102,7349220.0,22111046.0
W101,7303944.0,22110531.0
W107,7305911.0,22077963.0


In [31]:
df["Stock_Status"] = np.where(df["Closing_Stock"]<=0, "Stockout", "In Stock")

In [32]:
prod_path = Path.cwd().parent.parent/"Data"/"Cleaned"/"Products.csv"
products_df = pd.read_csv(prod_path)
products_df

,Product_ID,Product_Name,Category,Sub_Category,Unit_Cost,Selling_Price,Supplier_ID
0,P1000,Product 1,Office,Stationery,344.99,701.83,S131
1,P1001,Product 2,Beauty,Makeup,266.87,460.73,S108
2,P1002,Product 3,Toys,Plush,194.66,268.68,S139
3,P1003,Product 4,Fashion,Footwear,149.63,177.36,S119
4,P1004,Product 5,Office,Office supplies,210.06,288.91,S189
...,...,...,...,...,...,...,...
995,P1995,Product 996,Fmcg,Personal care,106.67,197.06,S110
996,P1996,Product 997,Fmcg,Personal care,130.52,177.40,S125
997,P1997,Product 998,Toys,Action toys,111.23,133.17,S141
998,P1998,Product 999,Home & kitchen,Appliances,188.99,242.41,S126


In [34]:
df["COGS"] = (df["Sold_Qty"] * products_df["Unit_Cost"]).round(2)
df["Avg_Inventory"] = ((df["Opening_Stock"]+df["Closing_Stock"])*products_df["Unit_Cost"]/2).round(2)
df["Inventory_Turnover"] = (df["COGS"]/df["Avg_Inventory"]).round(2)

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365010 entries, 0 to 365009
Data columns (total 14 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Inventory_ID        365010 non-null  object        
 1   Product_ID          365010 non-null  object        
 2   Warehouse_ID        365003 non-null  object        
 3   Date                363184 non-null  datetime64[ns]
 4   Opening_Stock       365010 non-null  float64       
 5   Received_Qty        365010 non-null  float64       
 6   Sold_Qty            365010 non-null  float64       
 7   Damaged_Qty         365010 non-null  float64       
 8   Closing_Stock       365010 non-null  float64       
 9   Month               363184 non-null  float64       
 10  Stock_Status        365010 non-null  object        
 11  COGS                1000 non-null    float64       
 12  Avg_Inventory       1000 non-null    float64       
 13  Inventory_Turnover  1000 non-

In [36]:
df.to_csv("../../Data/Processed/Inventory.csv", index=False)